# Kandinsky Game→Audio — Step 3: Patch and Dry-Run the Real Fused Block

This notebook applies a small, reversible patch to a **scratch copy** of Kandinsky's pinned `FusedTransformerDecoderBlock`.

It adds a gated Game→Audio cross-attention after the existing text and optional video/audio updates, before the audio FFN. It then exercises that real block in audio-only and joint video+audio calls using tiny random tensors.

**No model weights, training, or audio generation are used.** This is a block-level integration dry-run, not yet a change to Kandinsky's full pipeline or generation API.

## What changes in this step

- Keep Kandinsky's existing text→audio and video↔audio paths intact.
- Add an optional `game_tokens` input to the fused block.
- Let audio hidden states query game tokens using Kandinsky's own `MultiheadCrossAttention`.
- Use a zero-initialized, timestep-conditioned residual gate, so the new branch starts with no effect.
- Accept a separate key mask for missing or padded game tokens.

The block receives game tokens directly in this dry-run. The next integration step will pass them through the full `DiffusionTransformer3D` and pipeline interfaces.

In [ ]:
import subprocess
from pathlib import Path

import torch
import torch.nn.functional as F
print("Python:", __import__("sys").version.split()[0])
print("PyTorch:", torch.__version__)
print("Device: CPU-sized dry-run; no GPU is required.")

## 1. Fetch the exact Kandinsky source revision

The repository is public. We clone source only and check out the same pinned revision used in Steps 1 and 2.

In [ ]:
K6_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
K6_DIR = Path("/content/kandinsky-6-arch-lab")

if not K6_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/kandinskylab/kandinsky-6.git", str(K6_DIR)
    ], check=True)

current = subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip()
if current != K6_COMMIT:
    subprocess.run(
        ["git", "-C", str(K6_DIR), "fetch", "--depth", "1", "origin", K6_COMMIT],
        check=True
    )
    subprocess.run(
        ["git", "-C", str(K6_DIR), "checkout", "--detach", K6_COMMIT],
        check=True
    )

DIT_PATH = K6_DIR / "kandinsky/core/components/dit.py"
source = DIT_PATH.read_text(encoding="utf-8")
print("Pinned source:", subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip())

## 2. Patch a separate copy of the Fused block

The official `dit.py` stays untouched. We write the patched version to `dit_game.py` in the Colab runtime. The helper below requires every source anchor exactly once; if upstream structure differs, it stops instead of guessing where to insert code.

In [ ]:
def replace_once(text, old, new, label):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f"Expected one source anchor for {label}; found {count}.")
    return text.replace(old, new, 1)

patched = source

patched = replace_once(
    patched,
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n    ):\n",
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n"
    "        game_dim: int | None = None,\n    ):\n",
    "optional game feature dimension",
)

patched = replace_once(
    patched,
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n",
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n"
    "        self.game_dim      = game_dim\n"
    "        if game_dim is None:\n"
    "            self.game_audio_norm = None\n"
    "            self.game_to_audio = None\n"
    "            self.game_audio_modulation = None\n"
    "        else:\n"
    "            self.game_audio_norm = nn.LayerNorm(model_dim_a, elementwise_affine=False)\n"
    "            self.game_to_audio = MultiheadCrossAttention(model_dim_a, head_dim_a, game_dim, engine)\n"
    "            self.game_audio_modulation = Modulation(time_dim_a, model_dim_a, 1)\n",
    "Game→Audio modules",
)

patched = replace_once(
    patched,
    "        av_gate_scale: float = 1.0,\n        va_gate_scale: float = 1.0,\n    ):\n",
    "        av_gate_scale: float = 1.0,\n        va_gate_scale: float = 1.0,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n    ):\n",
    "optional game inputs on FusedTransformerDecoderBlock.forward",
)

patched = replace_once(
    patched,
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # ---- FFN ----\n",
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # Game events condition the audio stream in both audio-only and AV calls.\n"
    "        if game_tokens is not None:\n"
    "            if aud is None:\n"
    "                raise ValueError(\"Game conditioning requires an audio stream.\")\n"
    "            if self.game_to_audio is None:\n"
    "                raise ValueError(\"Construct this block with game_dim to accept game_tokens.\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[-1] != self.game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, tokens, game_dim].\")\n"
    "            if game_tokens.shape[0] != aud.shape[0]:\n"
    "                raise ValueError(\"game_tokens and audio must have the same batch size.\")\n"
    "            game_query = self.game_audio_norm(aud).type_as(aud)\n"
    "            game_update = self.game_to_audio(game_query, game_tokens, game_attn_mask)\n"
    "            game_gate = self.game_audio_modulation(t_a)\n"
    "            aud = apply_gate_sum(aud, game_update, game_gate).type_as(aud)\n"
    "\n"
    "        # ---- FFN ----\n",
    "Game→Audio insertion before the audio FFN",
)

PATCH_PATH = DIT_PATH.with_name("dit_game.py")
PATCH_PATH.write_text(patched, encoding="utf-8")
print("Scratch patch written:", PATCH_PATH)
print("Original dit.py unchanged:", DIT_PATH.exists())

## 3. Load the actual block without installing the full pipeline

Importing Kandinsky's package root also imports its generation pipeline and optional runtime packages. For this dry-run, we create a small Python package namespace and route the pinned attention interface to PyTorch SDPA. The block, its attention modules, normalization, modulation, and RoPE classes still come from the pinned Kandinsky source. This keeps setup short and avoids downloading weights or compiling optional kernels.

In [ ]:
import importlib.util
import sys
import types

if "kandinsky" in sys.modules:
    raise RuntimeError(
        "A Kandinsky package is already imported in this runtime. Restart the runtime, "
        "then run this notebook from the top to keep the source isolation clean."
    )

def register_package(name, path):
    module = types.ModuleType(name)
    module.__path__ = [str(path)]
    module.__package__ = name
    sys.modules[name] = module

register_package("kandinsky", K6_DIR / "kandinsky")
register_package("kandinsky.core", K6_DIR / "kandinsky/core")
register_package("kandinsky.core.components", K6_DIR / "kandinsky/core/components")
register_package("kandinsky.core.components.attention", K6_DIR / "kandinsky/core/components/attention")
register_package("kandinsky.runtime", K6_DIR / "kandinsky/runtime")
register_package("kandinsky.runtime.kernels", K6_DIR / "kandinsky/runtime/kernels")

def pinned_sdpa(q, k, v, attn_mask=None):
    # Same tensor layout and boolean-mask convention as Kandinsky's SDPA wrapper.
    q = q.transpose(1, 2).contiguous()
    k = k.transpose(1, 2).contiguous()
    v = v.transpose(1, 2).contiguous()
    return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask).transpose(1, 2).contiguous()

class SDPAOnlyEngine:
    def __init__(self, engine="auto"):
        if engine not in {"auto", "sdpa"}:
            raise ValueError("This dry-run intentionally supports only auto/sdpa.")
    def get_attention(self):
        return pinned_sdpa

engine_module = types.ModuleType("kandinsky.runtime.kernels.attention_engine")
engine_module._sdpa = pinned_sdpa
engine_module.SelfAttentionEngine = SDPAOnlyEngine
engine_module.resolve_attention_engine = lambda requested: "sdpa"
sys.modules[engine_module.__name__] = engine_module

def load_dit_module(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

base_dit = load_dit_module("kandinsky.core.components.dit_base", DIT_PATH)
game_dit = load_dit_module("kandinsky.core.components.dit_game", PATCH_PATH)
print("Loaded block source:", game_dit.__file__)
print("Block class:", game_dit.FusedTransformerDecoderBlock.__name__)

## 4. Create a tiny block and reuse the base weights

The small dimensions below are for a CPU-friendly code-path check; they are not Kandinsky checkpoint dimensions. We copy the base block's weights into the patched block. The only missing parameters should belong to the newly added Game→Audio branch.

In [ ]:
torch.manual_seed(23)

dims = dict(
    model_dim=24,
    time_dim=16,
    ff_dim=48,
    head_dim=6,
    model_dim_a=24,
    time_dim_a=16,
    ff_dim_a=48,
    head_dim_a=6,
    engine="sdpa",
)
GAME_DIM = 8

base_block = base_dit.FusedTransformerDecoderBlock(**dims).eval()
game_block = game_dit.FusedTransformerDecoderBlock(**dims, game_dim=GAME_DIM).eval()

load_result = game_block.load_state_dict(base_block.state_dict(), strict=False)
assert not load_result.unexpected_keys
print("New branch parameters awaiting fine-tuning:")
for name in load_result.missing_keys:
    print(" -", name)

## 5. Prepare synthetic inputs

The masks use PyTorch SDPA semantics: `True` means the key is visible to attention. Event tokens here are random placeholders, not output from a trained game encoder.

In [ ]:
batch = 2
video_length = 6
audio_length = 9
text_length = 4
game_length = 5
head_dim = dims["head_dim"]

vis = torch.randn(batch, video_length, dims["model_dim"])
aud = torch.randn(batch, audio_length, dims["model_dim_a"])
text_v = torch.randn(batch, text_length, dims["model_dim"])
text_a = torch.randn(batch, text_length, dims["model_dim_a"])
time_v = torch.randn(batch, dims["time_dim"])
time_a = torch.randn(batch, dims["time_dim_a"])
game_tokens = torch.randn(batch, game_length, GAME_DIM)
game_valid = torch.tensor([
    [True, True, True, False, False],
    [True, True, False, False, False],
])
game_attn_mask = game_valid[:, None, None, :]

rope_v = base_dit.RoPE1D(head_dim)(torch.arange(video_length))
rope_a = base_dit.RoPE1D(head_dim)(torch.arange(audio_length))
time_pair = (time_v, time_a)

print("Video hidden:", tuple(vis.shape))
print("Audio hidden:", tuple(aud.shape))
print("Game tokens:", tuple(game_tokens.shape))
print("Game attention mask:", tuple(game_attn_mask.shape))

## 6. Audio-only path: the closed gate preserves the original block

First call the original block with no video, then the patched block with game tokens. Since the new modulation is zero-initialized, both audio outputs should be identical. This checks that the branch also runs when the video stream is absent.

In [ ]:
with torch.no_grad():
    _, audio_base_only = base_block(
        None, aud, text_v, text_a, time_pair, None, rope_a
    )
    _, audio_game_closed = game_block(
        None, aud, text_v, text_a, time_pair, None, rope_a,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )

torch.testing.assert_close(audio_game_closed, audio_base_only, rtol=0, atol=0)
print("Audio-only output:", tuple(audio_game_closed.shape))
print("Zero-gate parity with base block: passed")

## 7. Joint video+audio path: preserve AV behavior, then allow game influence

With the gate closed, the patched block should preserve both outputs. We then open only the new game gate manually. This is a plumbing demonstration, not training: the gate value is chosen for the demo and has not been learned.

In [ ]:
with torch.no_grad():
    video_base, audio_base = base_block(
        vis, aud, text_v, text_a, time_pair, rope_v, rope_a
    )
    video_closed, audio_closed = game_block(
        vis, aud, text_v, text_a, time_pair, rope_v, rope_a,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )

torch.testing.assert_close(video_closed, video_base, rtol=0, atol=0)
torch.testing.assert_close(audio_closed, audio_base, rtol=0, atol=0)
print("Joint outputs:", tuple(video_closed.shape), tuple(audio_closed.shape))
print("Zero-gate parity with base AV block: passed")

with torch.no_grad():
    game_block.game_audio_modulation.out_layer.bias.fill_(0.1)
    video_open, audio_open = game_block(
        vis, aud, text_v, text_a, time_pair, rope_v, rope_a,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )

audio_delta = (audio_open - audio_base).abs().mean().item()
assert audio_delta > 0
torch.testing.assert_close(video_open, video_base, rtol=0, atol=0)
print("Mean audio change with demo gate:", audio_delta)
print("Video output remains unchanged by Game→Audio: passed")

## 8. Confirm that padded game tokens are ignored

We change only masked game-token values while keeping the event mask and all other inputs fixed. The resulting audio should stay the same.

In [ ]:
changed_game_tokens = game_tokens.clone()
changed_game_tokens[~game_valid] += 1000.0

with torch.no_grad():
    _, audio_masked_a = game_block(
        None, aud, text_v, text_a, time_pair, None, rope_a,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )
    _, audio_masked_b = game_block(
        None, aud, text_v, text_a, time_pair, None, rope_a,
        game_tokens=changed_game_tokens,
        game_attn_mask=game_attn_mask,
    )

torch.testing.assert_close(audio_masked_a, audio_masked_b, rtol=1e-5, atol=1e-6)
print("Padded-token isolation: passed")

## What this proves — and what it does not

**Demonstrated in Kandinsky's actual fused block:** the new branch can be placed before the audio FFN, receives audio queries and game keys/values, runs with or without video, preserves the original block when its gate is closed, and respects a separate game-token mask.

**Not demonstrated:** the full `DiffusionTransformer3D` forward API, pipeline/config wiring, compatibility with a real pretrained checkpoint, training, or audible control. The new parameters appear as missing keys when loading the old block state dict; that is expected and must be handled deliberately in a later step.

**Next:** propagate optional game tokens and their mask through `DiffusionTransformer3D.forward` and its single/fused block loops. Only after that path works should we inspect the real checkpoint loader and decide how to train the new branch.